# **Deep Thinking RAG Pipeline**

### **Defining State**

In [1]:
from pydantic import BaseModel, Field
from langchain_core.documents import Document
from typing import Literal, List, Optional
from langchain_core.output_parsers import StrOutputParser
import re

In [2]:
# Central Configuration Dictionary to manage all system parameters
config = {
    "data_dir": "../data",                           # Directory to store raw and cleaned data
    "vector_store_dir": "./vector_store",           # Directory to persist our vector store
    "llm_provider": "openai",                       # The LLM provider we are using
    "reasoning_llm": "gpt-4o",                      # The powerful model for planning and synthesis
    "fast_llm": "gpt-4o-mini",                      # A faster, cheaper model for simpler tasks like the baseline RAG
    "embedding_model": "text-embedding-3-small",    # The model for creating document embeddings
    "reranker_model": "cross-encoder/ms-marco-MiniLM-L-6-v2", # The model for precision reranking
    "max_reasoning_iterations": 7,                  # A safeguard to prevent the agent from getting into an infinite loop
    "top_k_retrieval": 10,                          # Number of documents for initial broad recall
    "top_n_rerank": 3,                              # Number of documents to keep after precision reranking
}

In [3]:
# importing required libraries
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

data_path_clean = os.path.join(config["data_dir"], "nvda_10k_2023_clean.txt")

# loadin text data
print('Loading and chunking documents...')
loader = TextLoader(data_path_clean, encoding='utf-8')
docs = loader.load()

# splitting text data into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
splitted_docs = text_splitter.split_documents(docs)

print(f"Documents loaded and split into {len(splitted_docs)} chunks.")

Loading and chunking documents...
Documents loaded and split into 571 chunks.


In [4]:



# defining state for single step in agent reasoning plan
class Step(BaseModel):
    sub_question: str = Field(description="A spesific answerable question for this step")
    justification: str = Field(description="Simple justification why this step is important for answering query")
    tool: Literal["search_10k", "search_web"] = Field(description="Tool to use for this step")
    keywords: List[str] = Field(description="A list of critical keywords for searching relevant document sections.")
    document_section: Optional[str] = Field(description="A likely document section title (e.g., 'Item 1A. Risk Factors') to search within. Only for 'search_10k' tool.")


# planning model
class Plan(BaseModel):
    plan:List[Step] = Field(description="Detailed multistep plan to answer user's query")


# model for storing past history of completed steps
class PastStep(BaseModel):
    step_index:str = Field(description="Index of each completed step e.g. 1, 2, 3")
    sub_question:str = Field(description="sub question ask in this step")
    retrieved_docs:List[Document] = Field(description="retrieved and ranked docs for this step")
    summary:str = Field(description="Agent's one sentence summary of findings from this step")


In [5]:
# main state dictionary that will pass among all nodes

class RAGState(BaseModel):
    original_question: str
    plan: Plan
    past_steps: List[PastStep]
    current_step_index: int
    retrieved_docs: List[Document]
    reranked_docs: List[Document]
    synthesized_context: str
    final_answer: str

### **Building each module of system**

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from rich.pretty import pprint as rprint

# The system prompt that instructs the LLM how to behave as a planner
planner_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert research planner. Your task is to create a clear, multi-step plan to answer a complex user query by retrieving information from multiple sources.
You have two tools available:
1. `search_10k`: Use this to search for information within NVIDIA's 2023 10-K financial filing. This is best for historical facts, financial data, and stated company policies or risks from that specific time period.
2. `search_web`: Use this to search the public internet for recent news, competitor information, or any topic that is not specific to NVIDIA's 2023 10-K.
Decompose the user's query into a series of simple, sequential sub-questions. For each step, decide which tool is more appropriate.
For `search_10k` steps, also identify the most likely section of the 10-K (e.g., 'Item 1A. Risk Factors', 'Item 7. Management's Discussion and Analysis...').
It is critical to use the exact section titles found in a 10-K filing where possible."""),
    ("human", "User Query: {question}") 
])

In [7]:
complex_query_adv = "Based on NVIDIA's 2023 10-K filing, identify their key risks related to competition. Then, find recent news (post-filing, from 2024) about AMD's AI chip strategy and explain how this new strategy directly addresses or exacerbates one of NVIDIA's stated risks."

In [9]:
# defining reasoning model
reasoning_llm = ChatOpenAI(model=config["reasoning_llm"], temperature=0)

# creating planner agent
planner_agent = planner_prompt | reasoning_llm.with_structured_output(Plan)

# testing planner agent on complex query
test_result = planner_agent.invoke({"question":complex_query_adv})

rprint(test_result)

c:\Users\masan\OneDrive\Desktop\Adaptive Multi-Agent RAG System\.venv\Lib\site-packages\pydantic\v1\main.py:1054: UserWarning: LangSmith now uses UUID v7 for run and trace identifiers. This warning appears when passing custom IDs. Please use: from langsmith import uuid7
            id = uuid7()
Future versions will require UUID v7.
  input_data = validator(cls_, input_data)


Plan(
│   plan=[
│   │   Step(
│   │   │   sub_question="What are NVIDIA's key risks related to competition as stated in their 2023 10-K filing?",
│   │   │   justification="Understanding NVIDIA's stated risks related to competition is essential to evaluate how AMD's strategies might impact them.",
│   │   │   tool='search_10k',
│   │   │   keywords=['competition', 'risks', 'competitive risks'],
│   │   │   document_section='Item 1A. Risk Factors'
│   │   ),
│   │   Step(
│   │   │   sub_question="What is AMD's AI chip strategy as of 2024?",
│   │   │   justification="Identifying AMD's current AI chip strategy will help in assessing its impact on NVIDIA's competitive risks.",
│   │   │   tool='search_web',
│   │   │   keywords=['AMD', 'AI chip strategy', '2024'],
│   │   │   document_section=None
│   │   ),
│   │   Step(
│   │   │   sub_question="How does AMD's 2024 AI chip strategy address or exacerbate NVIDIA's competitive risks?",
│   │   │   justification="This step will directly connect AMD's strategy to NVIDIA's risks, providing a comprehensive answer to the user's query.",
│   │   │   tool='search_web',
│   │   │   keywords=['AMD strategy impact on NVIDIA', 'NVIDIA competitive risks'],
│   │   │   document_section=None
│   │   )
│   ]
)

In [10]:
# Query rewriter prompt


# The prompt for our query rewriter, instructing it to act as a search expert
query_rewriter_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a search query optimization expert. Your task is to rewrite a given sub-question into a highly effective search query for a vector database or web search engine, using keywords and context from the research plan.
The rewritten query should be specific, use terminology likely to be found in the target source (a financial 10-K or news articles), and be structured to retrieve the most relevant text snippets."""),
    ("human", "Current sub-question: {sub_question}\n\nRelevant keywords from plan: {keywords}\n\nContext from past steps:\n{past_context}")
])

query_rewriter_agent = query_rewriter_prompt | reasoning_llm | StrOutputParser()
print("Query Rewriter Agent created successfully.")

print("\n--- Testing Query Rewriter Agent ---")

test_sub_q = "How does AMD's 2024 AI chip strategy potentially exacerbate the competitive risks identified in NVIDIA's 10-K?"
test_keywords = ['impact', 'threaten', 'competitive pressure', 'market share', 'technological change']

test_past_context = "Step 1 Summary: NVIDIA's 10-K lists intense competition and rapid technological change as key risks. Step 2 Summary: AMD launched its MI300X AI accelerator in 2024 to directly compete with NVIDIA's H100."

# Invoke the agent with our test data
rewritten_q = query_rewriter_agent.invoke({
    "sub_question": test_sub_q,
    "keywords": test_keywords,
    "past_context": test_past_context
})

print(f"Original sub-question: {test_sub_q}")
print(f"Rewritten Search Query: {rewritten_q}")

Query Rewriter Agent created successfully.

--- Testing Query Rewriter Agent ---
Original sub-question: How does AMD's 2024 AI chip strategy potentially exacerbate the competitive risks identified in NVIDIA's 10-K?
Rewritten Search Query: "AMD 2024 AI chip strategy impact on NVIDIA competitive risks 10-K MI300X vs H100 market share technological change"


In [11]:
# ── Step 1: Load the raw text ──────────────────────────────────────────────
raw_text = docs[0].page_content

# ── Step 2: Skip the Table of Contents ────────────────────────────────────
# The TOC ends near the "WHERE YOU CAN FIND MORE INFORMATION" landmark.
# We slice the text from that point so the TOC "Item X." entries are ignored.
toc_end_marker = "WHERE YOU CAN FIND MORE INFORMATION"
toc_end_pos = raw_text.find(toc_end_marker)
if toc_end_pos == -1:
    raise ValueError("Could not find TOC end marker in document.")
body_text = raw_text[toc_end_pos:]

# ── Step 3: Correct section pattern ───────────────────────────────────────
# In this document, headers appear as:  "ITEM 1. BUSINESS\n"
# i.e., all-caps, on a SINGLE line with the title. Pattern must match this.
# Use a capturing group so re.split returns headers as separate list elements.
section_pattern = r'(ITEM\s+\d+[A-Z]?\.\s+[^\n]+)'

# ── Step 4: Extract titles ─────────────────────────────────────────────────
section_titles = re.findall(section_pattern, body_text)
section_titles = [title.strip().replace('\n', ' ') for title in section_titles]

# ── Step 5: Split into content blocks ─────────────────────────────────────
# re.split with a capturing group returns: [pre, header, content, header, content, ...]
raw_parts = re.split(section_pattern, body_text)

# Even-indexed parts [0, 2, 4, ...] are content; odd-indexed [1, 3, 5, ...] are headers.
# parts[0] is the preamble before first section — skip it.
sections_content = [
    part.strip()
    for part in raw_parts[2::2]   # every content block, skipping preamble
    if part.strip()
]

# ── Step 6: Sanity check ───────────────────────────────────────────────────
print(f"Identified {len(section_titles)} document sections.")
assert len(section_titles) == len(sections_content), \
    f"Mismatch: {len(section_titles)} titles vs {len(sections_content)} content blocks"

Identified 22 document sections.


In [12]:
# creating metadata reach documents

import uuid

# path of data
data_path_clean = os.path.join(config["data_dir"], "nvda_10k_2023_clean.txt")

# text splitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

doc_chunks_with_metadata = []

for i, content in enumerate(sections_content):

    title = section_titles[i]
    chunks = text_splitter.split_text(content)

    for chunk in chunks:
        chunk_id = str(uuid.uuid4())
        doc_chunks_with_metadata.append(
            Document(
                page_content=chunk,
                metadata={
                    "section": title,
                    "source_doc": data_path_clean,
                    "id": chunk_id
                }
            )
        )

print(f"Created {len(doc_chunks_with_metadata)} chunks with section metadata.")
print("\n--- Sample Chunk with Metadata ---")

# To prove it worked, let's find a chunk that we know should be in the 'Risk Factors' section and print it
sample_chunk = next(
    c for c in doc_chunks_with_metadata
    if "risk factors" in c.metadata.get("section", "").lower()
)
print(sample_chunk)


Created 541 chunks with section metadata.

--- Sample Chunk with Metadata ---
page_content='In evaluating NVIDIA, the following risk factors should be considered in addition to the other information in this Annual Report on Form 10-K. Purchasing or owning NVIDIA common stock involves investment risks including, but not limited to, the risks described below. Any one of the following risks could harm our business, financial condition, results of operations or reputation, which could cause our stock price to decline, and you may lose all or a part of your investment. Additional risks, trends and uncertainties not presently known to us or that we currently believe are immaterial may also harm our business, financial condition, results of operations or reputation.

 Risk Factors Summary

 Risks Related to Our Industry and Markets

 •Failure to meet the evolving needs of our industry and markets may adversely impact our financial results.

 •

 Failure to meet the evolving needs of our indus

In [13]:
# creating supervisor agent to choose retrieval strategy

class RetrievalDecision(BaseModel):

    strategy: Literal["vector_search", "keyword_search", "hybrid_search"]
    justification: str


retrieval_supervisor_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a retrieval strategy expert. Based on the user's query, you must decide the best retrieval strategy.
You have three options:
1. `vector_search`: Best for conceptual, semantic, or similarity-based queries.
2. `keyword_search`: Best for queries with specific, exact terms, names, or codes (e.g., 'Item 1A', 'Hopper architecture').
3. `hybrid_search`: A good default that combines both, but may be less precise than a targeted strategy."""),
    ("human", "User Query: {sub_question}") # The rewritten search query will be passed here.
])

retrieval_supervisor_agent = retrieval_supervisor_prompt | reasoning_llm.with_structured_output(RetrievalDecision)
print("Retrieval Supervisor Agent created.")

print("\n--- Testing Retrieval Supervisor Agent ---")
query1 = "revenue growth for the Compute & Networking segment in fiscal year 2023"
decision1 = retrieval_supervisor_agent.invoke(query1)
print(f"Query: '{query1}'")
print(f"Decision: {decision1.strategy}, Justification: {decision1.justification}")

query2 = "general sentiment about market competition and technological innovation"
decision2 = retrieval_supervisor_agent.invoke({"sub_question": query2})
print(f"\nQuery: '{query2}'")
print(f"Decision: {decision2.strategy}, Justification: {decision2.justification}")

Retrieval Supervisor Agent created.

--- Testing Retrieval Supervisor Agent ---
Query: 'revenue growth for the Compute & Networking segment in fiscal year 2023'
Decision: keyword_search, Justification: The query is looking for specific information related to 'revenue growth', 'Compute & Networking segment', and 'fiscal year 2023'. These are precise terms and likely to be found in financial reports or documents where exact matches are important. Therefore, a keyword search is the most appropriate strategy to retrieve accurate and relevant information.

Query: 'general sentiment about market competition and technological innovation'
Decision: vector_search, Justification: The query is conceptual and seeks an understanding of general sentiment, which involves semantic interpretation and similarity-based retrieval rather than specific terms or codes.


### **Creating Retrieval Strategy**

In [14]:
import numpy as np
from rank_bm25 import BM25Okapi
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

# embedding function
embedding_function = OpenAIEmbeddings(model=config['embedding_model'])


print("Creating advanced vector store")

advanced_vector_store = Chroma.from_documents(
    documents=doc_chunks_with_metadata,
    embedding=embedding_function
)

print(f"Advanced vector store created with {advanced_vector_store._collection.count()} embeddings.")

Creating advanced vector store
Advanced vector store created with 541 embeddings.


In [18]:
print("Bulding BM25 Index keyword search")

tokenized_corpus = [doc.page_content.split(" ") for doc in doc_chunks_with_metadata]

doc_ids = [doc.metadata["id"] for doc in doc_chunks_with_metadata]

doc_map = {doc.metadata["id"]:doc for doc in doc_chunks_with_metadata}

bm25 = BM25Okapi(tokenized_corpus)


Bulding BM25 Index keyword search


In [40]:
# defining retrieval tools

# vector search only
def vector_search(query:str, section_filter:str=None, k:int=10):
    filter_dict = {'section': section_filter} if section_filter and "Unknown" not in section_filter else None
    return advanced_vector_store.similarity_search(query=query, filter=filter_dict, k=k)

# keyword search only
def bm25_search(query:str, k:int=10):
    tokenized_query = query.split(" ")
    scores = bm25.get_scores(tokenized_query)

    sorted_scores = np.argsort(scores)[::-1][:k]

    return [doc_map[doc_ids[id]] for id in sorted_scores]

# hybrid search
def hybrid_search(query:str, section_filter:str=None, k:int=10):

    semantic_doc = vector_search(query, section_filter, k)
    bm25_docs = bm25_search(query, k)

    all_docs  = {docs.metadata["id"]:docs for docs in semantic_doc+bm25_docs}.values()

    ranked_list = [[doc.metadata["id"] for doc in semantic_doc], [doc.metadata["id"] for doc in bm25_docs]]

    rff_scores = {}
    
    for doc_list in ranked_list:
        for i, doc in enumerate(doc_list):
            if doc not in rff_scores:
                rff_scores[doc] = 0

            rff_scores[doc] += 1/(i+61)
    
    sorted_doc_ids = sorted(rff_scores.keys(), key=lambda x:rff_scores[x], reverse=True)
    final_doc = [doc_map[doc_id] for doc_id in sorted_doc_ids[:k]]

    return final_doc


In [41]:
hybrid_search("Based on NVIDIA's 2023 10-K filing, identify their key risks related to competition.")


[Document(metadata={'section': 'ITEM 1A. RISK FACTORS', 'source_doc': '../data\\nvda_10k_2023_clean.txt', 'id': '904fc3c7-9b96-401a-83c4-8305f8761e28'}, page_content='In evaluating NVIDIA, the following risk factors should be considered in addition to the other information in this Annual Report on Form 10-K. Purchasing or owning NVIDIA common stock involves investment risks including, but not limited to, the risks described below. Any one of the following risks could harm our business, financial condition, results of operations or reputation, which could cause our stock price to decline, and you may lose all or a part of your investment. Additional risks, trends and uncertainties not presently known to us or that we currently believe are immaterial may also harm our business, financial condition, results of operations or reputation.\n\n Risk Factors Summary\n\n Risks Related to Our Industry and Markets\n\n •Failure to meet the evolving needs of our industry and markets may adversely im

In [37]:
doc_map[doc_ids[178]]

Document(metadata={'section': 'ITEM 1A. RISK FACTORS', 'source_doc': '../data\\nvda_10k_2023_clean.txt', 'id': 'b7c06bf3-9748-42db-a63a-88acd05fc022'}, page_content='•exposure to additional cybersecurity risks and vulnerabilities;\n\n •\n\n exposure to additional cybersecurity risks and vulnerabilities;\n\n •potential failure of our due diligence processes to identify significant issues with the assets or company in which we are investing or are acquiring; and\n\n •\n\n potential failure of our due diligence processes to identify significant issues with the assets or company in which we are investing or are acquiring; and\n\n •impairment of relationships with, or loss of our or our target’s employees, vendors and customers.\n\n •\n\n impairment of relationships with, or loss of our or our target’s employees, vendors and customers.')